# Feature Engineering & Selection for Machine Learning

**Junior Design **

---

## Why This Matters

You've now learned several ML algorithms — linear/logistic regression, decision trees, random forests, gradient descent, and ensemble methods. You might expect that picking the *right algorithm* is the most important decision in an ML project.

**It's not.** In practice, the quality of your *features* often matters far more than your choice of model.

> *"Coming up with features is difficult, time-consuming, requires expert knowledge. Applied machine learning is basically feature engineering."*  
> — Andrew Ng

As you begin working with your own datasets for your projects, you'll quickly discover that raw data is messy. Columns have missing values, categories are stored as text, features live on wildly different scales, and many columns may be irrelevant or redundant.

**Feature engineering** is the process of transforming raw data into features that better represent the underlying problem to the ML model, resulting in improved accuracy.

###  Agenda

| Topic | What We'll Think About|
|---|---|
| 1. Exploring Raw Data | Understanding what you're working with |
| 2. Handling Missing Data | Strategies for incomplete datasets |
| 3. Encoding Categorical Variables | Converting text to numbers |
| 4. Feature Scaling | Normalization vs. Standardization |
| 5. Feature Creation | Building new features from existing ones |
| 6. Dimensionality Reduction (PCA) | Compressing your feature space |
| 7. Feature Selection | Choosing the features that matter |
| 8. Putting It All Together | A complete pipeline |

---

## Setup

Let's import our libraries and set up our environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, LabelEncoder, 
    OneHotEncoder, PolynomialFeatures, OrdinalEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import (
    SelectKBest, f_regression, mutual_info_regression
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import fetch_openml

import warnings
warnings.filterwarnings('ignore')

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("All imports successful!")

---

## Part 1: Exploring Raw Data — The Ames Housing Dataset

We'll use the **Ames Housing Dataset**, which contains information about 1,460 house sales in Ames, Iowa. The goal is to predict the **sale price** of a house based on 79 features.

This is a classic dataset for feature engineering because it has:
- A mix of numerical and categorical features
- Meaningful missing values (not just random gaps)
- Features on very different scales
- Redundant and irrelevant features
- Ordinal categories (quality ratings) alongside nominal categories (neighborhood names)

In [ ]:
# Download the Ames Housing dataset from OpenML
ames = fetch_openml(name="house_prices", as_frame=True, parser='auto')
df = ames.frame.copy()

print(f"Dataset shape: {df.shape}")
print(f"\nColumn types:")
print(f"  Numeric:     {df.select_dtypes(include='number').shape[1]}")
print(f"  Categorical: {df.select_dtypes(include=['object','category']).shape[1]}")
print(f"\nTarget: SalePrice")
print(f"  Mean:   ${df['SalePrice'].mean():,.0f}")
print(f"  Median: ${df['SalePrice'].median():,.0f}")
print(f"  Range:  ${df['SalePrice'].min():,.0f} - ${df['SalePrice'].max():,.0f}")
df.head()

In [ ]:
# Overview of numeric features
print("=" * 60)
print("BASIC STATISTICS (Numeric Features)")
print("=" * 60)
df.describe().round(1)

In [ ]:
# ============================================================
# Check for missing values — ALWAYS your first step
# ============================================================
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

print("Columns with missing values:")
print("=" * 50)
print(missing_summary[missing_summary['Missing Count'] > 0].to_string())
print(f"\nTotal: {(missing > 0).sum()} columns have missing data")
print(f"Total missing cells: {df.isnull().sum().sum()} "
      f"out of {df.shape[0] * df.shape[1]} ({df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100:.1f}%)")

### Observations

This is a **real dataset** with **real messiness**. Notice a few things:

**Scale differences:** `LotArea` ranges from ~1,000 to 200,000+ while `OverallQual` goes from 1 to 10. Some ML algorithms (linear regression, SVMs, neural networks) are **very sensitive** to this.

**Missing values have meaning:** In this dataset, many missing values aren't random errors — they mean "this house doesn't have that feature":
- `PoolQC` is missing for ~99% of houses → most houses don't have a pool
- `Alley` is missing for ~94% → most houses don't have alley access
- `Fence` is missing for ~81% → most houses don't have a fence
- `GarageType/GarageFinish/etc.` are missing for the same ~81 houses → no garage
- `LotFrontage` (~18% missing) is more likely a data collection gap

**This distinction matters enormously** for how you handle the missing data.

---

## Part 2: Handling Missing Data

Missing data is nearly universal in real-world datasets. You have several strategies:

| Strategy | When to Use | Pros | Cons |
|---|---|---|---|
| **Drop rows** | Very few missing values | Simple | Lose data |
| **Drop columns** | Column is mostly missing (>50%) | Simple | Lose a feature |
| **Mean/Median imputation** | Numeric, roughly symmetric | Preserves dataset size | Reduces variance |
| **Mode imputation** | Categorical features | Simple | Can introduce bias |
| **Fill with "None"/0** | Missing = absence of feature | Most accurate for this case | Requires domain knowledge |


**Always ask: WHY is this data missing?**

In [ ]:
df_clean = df.copy()

# ============================================================
# Strategy 1: Drop columns that are almost entirely missing
# These columns have >80% missing and are unlikely to help
# ============================================================
drop_cols = [col for col in df_clean.columns 
             if df_clean[col].isnull().sum() / len(df_clean) > 0.80]

print("Dropping columns with >80% missing:")
for col in drop_cols:
    pct = df_clean[col].isnull().sum() / len(df_clean) * 100
    print(f"  {col:<15s} ({pct:.0f}% missing)")

df_clean = df_clean.drop(columns=drop_cols)
print(f"\nShape after dropping: {df_clean.shape}")

In [ ]:
# ============================================================
# Strategy 2: Domain-specific fill — "missing" means "none"
# ============================================================
# For garage-related columns, missing means "no garage"
# Let's verify: do the missing garage columns line up?

garage_cols = [c for c in df_clean.columns if 'Garage' in c]
print("Missing values in garage columns:")
for col in garage_cols:
    print(f"  {col:<15s} {df_clean[col].isnull().sum():>4d} missing")

print("\n→ These are (mostly) the same ~81 houses — they have no garage.")

In [ ]:
# Fill garage categoricals with 'None', garage numerics with 0
garage_cat_cols = df_clean[garage_cols].select_dtypes(include=['object', 'category']).columns
garage_num_cols = df_clean[garage_cols].select_dtypes(include='number').columns

df_clean[garage_cat_cols] = df_clean[garage_cat_cols].fillna('None')
df_clean[garage_num_cols] = df_clean[garage_num_cols].fillna(0)

# Same logic for basement columns — missing means no basement
bsmt_cols = [c for c in df_clean.columns if 'Bsmt' in c]
bsmt_cat_cols = df_clean[bsmt_cols].select_dtypes(include=['object', 'category']).columns
bsmt_num_cols = df_clean[bsmt_cols].select_dtypes(include='number').columns

df_clean[bsmt_cat_cols] = df_clean[bsmt_cat_cols].fillna('None')
df_clean[bsmt_num_cols] = df_clean[bsmt_num_cols].fillna(0)

# FireplaceQu — missing means no fireplace
if 'FireplaceQu' in df_clean.columns:
    df_clean['FireplaceQu'] = df_clean['FireplaceQu'].fillna('None')

# Fence — missing means no fence
if 'Fence' in df_clean.columns:
    df_clean['Fence'] = df_clean['Fence'].fillna('None')

# MasVnrType / MasVnrArea — missing likely means no masonry veneer
if 'MasVnrType' in df_clean.columns:
    df_clean['MasVnrType'] = df_clean['MasVnrType'].fillna('None')
if 'MasVnrArea' in df_clean.columns:
    df_clean['MasVnrArea'] = df_clean['MasVnrArea'].fillna(0)

print("After domain-specific fills:")
remaining = df_clean.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) > 0:
    print(remaining.to_string())
else:
    print("  No missing values remain!")
print(f"\nTotal missing cells remaining: {df_clean.isnull().sum().sum()}")

In [ ]:
# ============================================================
# Strategy 3: Median imputation for remaining numeric gaps
# ============================================================
# LotFrontage is likely a data collection gap — impute with
# neighborhood median (smarter than global median, since lot
# frontage depends on the neighborhood)

if 'LotFrontage' in df_clean.columns and df_clean['LotFrontage'].isnull().sum() > 0:
    print("LotFrontage: imputing with neighborhood median")
    df_clean['LotFrontage'] = df_clean.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median())
    )
    # If any neighborhoods were entirely missing, fall back to global median
    df_clean['LotFrontage'] = df_clean['LotFrontage'].fillna(df_clean['LotFrontage'].median())

# For any other remaining numeric columns, use global median
num_cols_missing = df_clean.select_dtypes(include='number').columns[
    df_clean.select_dtypes(include='number').isnull().any()
].tolist()

if num_cols_missing:
    print(f"Median imputation for: {num_cols_missing}")
    imputer_num = SimpleImputer(strategy='median')
    df_clean[num_cols_missing] = imputer_num.fit_transform(df_clean[num_cols_missing])

print(f"Numeric missing remaining: {df_clean.select_dtypes(include='number').isnull().sum().sum()}")

In [ ]:
# ============================================================
# Strategy 4: Mode imputation for remaining categorical gaps
# ============================================================
cat_cols_missing = df_clean.select_dtypes(include=['object', 'category']).columns[
    df_clean.select_dtypes(include=['object', 'category']).isnull().any()
].tolist()

if cat_cols_missing:
    print(f"Mode imputation for: {cat_cols_missing}")
    for col in cat_cols_missing:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(f"\n Total missing values remaining: {df_clean.isnull().sum().sum()}")

### Quick Check-In

We handled missing values using four different strategies based on *why* the data was missing:

1. **Dropped** columns that were almost entirely empty (>80% missing)
2. **Filled with "None"/0** where missing meant "feature doesn't exist" (garage, basement, fireplace)
3. **Neighborhood-median imputation** for `LotFrontage` (smarter than global median)
4. **Mode imputation** for remaining categorical gaps

> **For your projects:** Before you impute anything, look at the missing data patterns. Does the missingness tell you something? Could it even be a feature itself? (e.g., an `HasGarage` binary column derived from whether garage data was missing.)

---

## Part 3: Encoding Categorical Variables

ML models work with **numbers**. Categorical features like `Neighborhood` or `HouseStyle` need to be converted. There are several approaches, and the right choice depends on the *type* of category:

### Ordinal Encoding
Assign each category an integer that respects the natural order: `Poor=1, Fair=2, Good=3, Excellent=4`

**Use when:** Categories have a natural order. The Ames dataset has several of these: `ExterQual`, `BsmtQual`, `KitchenQual`, etc. all use the scale `Po < Fa < TA < Gd < Ex`.

### One-Hot Encoding
Create a binary column for each category:

| Nbr_OldTown | Nbr_Edwards | Nbr_NAmes | ... |
|---|---|---|---|
| 1 | 0 | 0 | ... |
| 0 | 1 | 0 | ... |

**Use when:** Categories are nominal — no natural order (e.g., `Neighborhood`, `SaleCondition`).

In [ ]:
# Let's survey our categorical columns
cat_cols = df_clean.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Total categorical columns: {len(cat_cols)}\n")

# Classify them: ordinal (quality/condition ratings) vs nominal
# The Ames dataset uses a common quality scale: None, Po, Fa, TA, Gd, Ex
quality_scale = {'None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'}

ordinal_candidates = []
nominal_candidates = []

for col in cat_cols:
    unique_vals = set(df_clean[col].dropna().unique())
    if unique_vals.issubset(quality_scale):
        ordinal_candidates.append(col)
    else:
        nominal_candidates.append(col)

print("ORDINAL (quality/condition scale — can be label encoded):")
for col in ordinal_candidates:
    print(f"  {col:<20s} values: {sorted(df_clean[col].unique())}")

print(f"\nNOMINAL (no natural order — should be one-hot encoded):")
for col in nominal_candidates:
    n = df_clean[col].nunique()
    print(f"  {col:<20s} {n:>2d} unique values")

In [ ]:
# ============================================================
# Ordinal Encoding for quality/condition features
# ============================================================
quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

df_encoded = df_clean.copy()

for col in ordinal_candidates:
    df_encoded[col] = df_encoded[col].map(quality_map)
    print(f"  {col}: mapped to 0-5 scale")

print(f"\n Ordinal encoding preserves the meaningful order:")
print(f"   None(0) < Po(1) < Fa(2) < TA(3) < Gd(4) < Ex(5)")

In [ ]:
# ============================================================
# One-Hot Encoding for nominal features
# ============================================================
# First, let's see which nominal columns have few vs many categories
print("Nominal columns by cardinality:")
for col in sorted(nominal_candidates, key=lambda c: df_encoded[c].nunique()):
    print(f"  {col:<20s} {df_encoded[col].nunique():>2d} categories")

# One-hot encode — using drop_first to avoid the dummy variable trap
shape_before = df_encoded.shape
df_encoded = pd.get_dummies(df_encoded, columns=nominal_candidates, drop_first=True)

print(f"\nShape before one-hot encoding: {shape_before}")
print(f"Shape after one-hot encoding:  {df_encoded.shape}")
print(f"Added {df_encoded.shape[1] - shape_before[1] + len(nominal_candidates)} new binary columns")

In [ ]:
# Quick look at some of the one-hot encoded neighborhood columns
nbr_cols = [c for c in df_encoded.columns if c.startswith('Neighborhood_')]
print(f"Neighborhood one-hot columns ({len(nbr_cols)} columns):")
print(df_encoded[['GrLivArea', 'SalePrice'] + nbr_cols[:5]].head(8).to_string())

### Why `drop_first=True`?

If a house is NOT in any of the displayed neighborhoods, it must be in the *dropped* one (the reference category). This avoids **multicollinearity** — a situation where one feature can be perfectly predicted from others. This is called the **dummy variable trap**, and it's especially problematic for linear models.

For tree-based models (Random Forest, Gradient Boosting), `drop_first` is less critical, but it's good practice.

---

## Part 4: Feature Scaling

This is one of the most impactful (and most overlooked) preprocessing steps.

### Why Scale?

Consider predicting SalePrice with two features:
- `LotArea`: ranges from ~1,300 to 215,000
- `OverallQual`: ranges from 1 to 10

In gradient descent, `LotArea` will dominate the gradient updates simply because its values are larger — **not** because it's more important. Scaling puts all features on equal footing.

### Two Common Approaches

**1. Standardization (Z-score normalization):**
$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$
Result: mean = 0, std = 1. Works well when data is roughly Gaussian.

**2. Min-Max Normalization:**
$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$
Result: values squeezed to [0, 1]. Useful when you need bounded values (e.g., neural networks).

### Which algorithms care about scaling?

| Needs Scaling | Doesn't Need Scaling |
|---|---|
| Linear/Logistic Regression | Decision Trees |
| SVMs | Random Forests |
| Neural Networks | Gradient Boosted Trees |
| K-Nearest Neighbors | |
| PCA | |

In [ ]:
# Key numeric features we'll focus on for demonstrations
key_numeric = ['LotArea', 'GrLivArea', 'YearBuilt', 'OverallQual',
               'TotalBsmtSF', 'BedroomAbvGr', 'FullBath', 'GarageCars', 'GarageArea']

# Visualize their raw distributions
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.flat, key_numeric):
    ax.hist(df_encoded[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('Count')
    ax.axvline(df_encoded[col].mean(), color='red', linestyle='--', 
               label=f'Mean={df_encoded[col].mean():.0f}')
    ax.legend(fontsize=8)
plt.suptitle('Distribution of Raw (Unscaled) Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Apply both scaling methods and compare
# ============================================================
scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()

df_standardized = df_encoded.copy()
df_normalized = df_encoded.copy()

df_standardized[key_numeric] = scaler_standard.fit_transform(df_encoded[key_numeric])
df_normalized[key_numeric] = scaler_minmax.fit_transform(df_encoded[key_numeric])

comparison = pd.DataFrame({
    'Feature': key_numeric,
    'Raw_Mean': [df_encoded[c].mean() for c in key_numeric],
    'Raw_Std': [df_encoded[c].std() for c in key_numeric],
    'Std_Mean': [df_standardized[c].mean() for c in key_numeric],
    'Std_Std': [df_standardized[c].std() for c in key_numeric],
    'MM_Min': [df_normalized[c].min() for c in key_numeric],
    'MM_Max': [df_normalized[c].max() for c in key_numeric],
}).round(3)

print(comparison.to_string(index=False))

In [ ]:
# Visualize: Before vs After scaling (pick 3 features with very different scales)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

demo_cols = ['LotArea', 'OverallQual', 'FullBath']
colors = ['#e74c3c', '#2ecc71', '#3498db']

# Raw
for col, color in zip(demo_cols, colors):
    axes[0].hist(df_encoded[col], bins=30, alpha=0.5, label=col, color=color)
axes[0].set_title('Raw Features', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].set_xlabel('Value')

# Standardized
for col, color in zip(demo_cols, colors):
    axes[1].hist(df_standardized[col], bins=30, alpha=0.5, label=col, color=color)
axes[1].set_title('After Standardization (Z-score)', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].set_xlabel('Standard deviations from mean')

# Min-Max
for col, color in zip(demo_cols, colors):
    axes[2].hist(df_normalized[col], bins=30, alpha=0.5, label=col, color=color)
axes[2].set_title('After Min-Max Normalization', fontsize=13, fontweight='bold')
axes[2].legend()
axes[2].set_xlabel('Value (0 to 1)')

plt.tight_layout()
plt.show()

print("Notice how all three features now overlap in the same range after scaling!")

In [ ]:
# ============================================================
# Impact of scaling on Linear Regression coefficients
# ============================================================
X_raw = df_encoded[key_numeric].copy()
X_scaled = df_standardized[key_numeric].copy()
y = df_encoded['SalePrice']

lr_raw = LinearRegression().fit(X_raw, y)
lr_scaled = LinearRegression().fit(X_scaled, y)

coef_comparison = pd.DataFrame({
    'Feature': key_numeric,
    'Coef (Raw)': lr_raw.coef_.round(2),
    'Coef (Standardized)': lr_scaled.coef_.round(2)
}).sort_values('Coef (Standardized)', key=abs, ascending=False)

print("Linear Regression Coefficients:")
print("=" * 55)
print(coef_comparison.to_string(index=False))
print("\n With standardized features, coefficients directly tell you")
print("   feature importance: larger |coefficient| = more important.")
print("   With raw features, you can't compare coefficients meaningfully.")

### Fit on Train, Transform on Test!

A very common mistake is fitting the scaler on the *entire* dataset, then splitting into train/test. This causes **data leakage** — your test set statistics leak into the training process.

```python
# data leakage!
X_scaled = scaler.fit_transform(X)  # fit on ALL data
X_train, X_test = train_test_split(X_scaled)

# no leakage
X_train, X_test = train_test_split(X)  # split FIRST
scaler.fit(X_train)                     # fit on train ONLY
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)  # transform test with train's params
```

We'll see how `Pipeline` makes this automatic in Part 8.

---

## Part 5: Feature Creation

Sometimes the most powerful features don't exist in your raw data — you have to **create** them. This is where domain knowledge becomes your superpower.

Common strategies:
1. **Derived features:** Compute something new (e.g., `age = current_year - year_built`)
2. **Interaction features:** Combine two features (e.g., `area × quality`)
3. **Aggregation features:** Combine related columns (e.g., `total_SF = 1stFlr + 2ndFlr + Bsmt`)
4. **Polynomial features:** Add squared/cubic terms for nonlinear relationships
5. **Log transforms:** Tame skewed distributions

In [ ]:
# ============================================================
# Hand-crafted features using domain knowledge about houses
# ============================================================
df_featured = df_encoded.copy()

# --- Derived features ---
# Age is more intuitive than raw year, and captures depreciation
df_featured['HouseAge'] = df_featured['YrSold'] - df_featured['YearBuilt']

# Years since last remodel (0 if never remodeled, since YearRemodAdd == YearBuilt)
df_featured['YearsSinceRemod'] = df_featured['YrSold'] - df_featured['YearRemodAdd']

# --- Aggregation features ---
# Total square footage (above ground + basement)
df_featured['TotalSF'] = df_featured['GrLivArea'] + df_featured['TotalBsmtSF']

# Total bathrooms (full + half, weighting half baths as 0.5)
df_featured['TotalBath'] = (df_featured['FullBath'] + 
                             0.5 * df_featured['HalfBath'] + 
                             df_featured['BsmtFullBath'] + 
                             0.5 * df_featured['BsmtHalfBath'])

# Total porch area
porch_cols = ['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
existing_porch = [c for c in porch_cols if c in df_featured.columns]
df_featured['TotalPorchSF'] = df_featured[existing_porch].sum(axis=1)

# --- Interaction features ---
# Quality x size captures "quality-adjusted" living space
df_featured['QualityArea'] = df_featured['OverallQual'] * df_featured['GrLivArea']

# --- Binary indicators ---
df_featured['HasGarage'] = (df_featured['GarageArea'] > 0).astype(int)
df_featured['HasBsmt'] = (df_featured['TotalBsmtSF'] > 0).astype(int)
df_featured['WasRemodeled'] = (df_featured['YearRemodAdd'] != df_featured['YearBuilt']).astype(int)

# --- Log transform of skewed features ---
df_featured['LogLotArea'] = np.log1p(df_featured['LotArea'])
df_featured['LogGrLivArea'] = np.log1p(df_featured['GrLivArea'])

# Summarize
new_features = ['HouseAge', 'YearsSinceRemod', 'TotalSF', 'TotalBath', 
                'TotalPorchSF', 'QualityArea', 'HasGarage', 'HasBsmt', 
                'WasRemodeled', 'LogLotArea', 'LogGrLivArea']

print("New engineered features:")
print("=" * 60)
for feat in new_features:
    print(f"  {feat:<20s} mean={df_featured[feat].mean():>10.1f}  "
          f"min={df_featured[feat].min():>8.1f}  max={df_featured[feat].max():>10.1f}")

In [ ]:
# ============================================================
# Do the new features help? Let's test!
# ============================================================
y = df_featured['SalePrice']

# Original numeric features only
X_original = df_featured[key_numeric]

# With our new features added
X_enhanced = df_featured[key_numeric + new_features]

lr = LinearRegression()

scores_original = cross_val_score(lr, X_original, y, cv=5, scoring='r2')
scores_enhanced = cross_val_score(lr, X_enhanced, y, cv=5, scoring='r2')

print("Linear Regression Performance (5-fold CV):")
print(f"  Original {len(key_numeric)} features:              R² = {scores_original.mean():.4f} ± {scores_original.std():.4f}")
print(f"  With {len(new_features)} engineered features added: R² = {scores_enhanced.mean():.4f} ± {scores_enhanced.std():.4f}")

improvement = scores_enhanced.mean() - scores_original.mean()
print(f"\n{'Improvement!' if improvement > 0 else 'No Improvement!'} R² changed by {improvement:+.4f}")

In [ ]:
# ============================================================
# Polynomial Features — automated feature creation
# ============================================================
X_small = df_featured[['GrLivArea', 'OverallQual']].values[:5]

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_small)

print("Original 2 features (first 5 rows):")
print(f"  {'GrLivArea':>10s}  {'OverallQual':>12s}")
for row in X_small:
    print(f"  {row[0]:>10.0f}  {row[1]:>12.0f}")

print(f"\nAfter PolynomialFeatures(degree=2):")
feature_names = poly.get_feature_names_out(['GrLivArea', 'OverallQual'])
print(f"  Features: {list(feature_names)}")
print(f"  Shape: {X_small.shape} → {X_poly.shape}")
print(f"\n  With degree=2 and {len(key_numeric)} input features, you'd get "
      f"{PolynomialFeatures(degree=2, include_bias=False).fit_transform(np.zeros((1, len(key_numeric)))).shape[1]} features!")
print(f"   This can lead to the 'curse of dimensionality.'")

### Takeaway on Feature Creation

The **best** engineered features come from understanding your problem domain. For the nuclear applications you'll encounter with pyMAISE, this means understanding what physical relationships exist between your input parameters and the quantities you're trying to predict.

For example, if you're predicting reactor power and you have both neutron flux and cross-section as features, their *product* (reaction rate) is a natural engineered feature that captures the underlying physics.

---

## Part 6: Dimensionality Reduction with PCA

**Principal Component Analysis (PCA)** finds the directions of maximum variance in your data and projects it onto a lower-dimensional space.

**Why use PCA?**
- Reduce computation time (fewer features)
- Remove multicollinearity
- Visualize high-dimensional data
- Sometimes improves model performance by removing noise

**Key requirement:** PCA requires **scaled** data (it's based on variance, so features with large scales will dominate).

In [ ]:
# ============================================================
# PCA on the Ames data
# ============================================================
all_numeric = df_featured.select_dtypes(include=[np.number]).drop(
    columns=['SalePrice', 'Id'], errors='ignore'
)

scaler = StandardScaler()
X_scaled_all = scaler.fit_transform(all_numeric)

print(f"Number of numeric features: {X_scaled_all.shape[1]}")

# Fit PCA with all components to see explained variance
pca_full = PCA()
pca_full.fit(X_scaled_all)

# Plot explained variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_show = min(40, len(pca_full.explained_variance_ratio_))

axes[0].bar(range(1, n_show + 1), 
            pca_full.explained_variance_ratio_[:n_show], 
            color='steelblue', edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Variance Explained by Each Component', fontweight='bold')

cumulative = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(range(1, len(cumulative) + 1), cumulative, 'o-', 
             color='steelblue', linewidth=2, markersize=3)
axes[1].axhline(y=0.90, color='red', linestyle='--', label='90% threshold')
axes[1].axhline(y=0.95, color='orange', linestyle='--', label='95% threshold')

n_90 = np.argmax(cumulative >= 0.90) + 1
n_95 = np.argmax(cumulative >= 0.95) + 1
axes[1].axvline(x=n_90, color='red', linestyle=':', alpha=0.5)
axes[1].axvline(x=n_95, color='orange', linestyle=':', alpha=0.5)

axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Variance Explained', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Components needed for 90% variance: {n_90} (out of {X_scaled_all.shape[1]})")
print(f"Components needed for 95% variance: {n_95} (out of {X_scaled_all.shape[1]})")

In [ ]:
# ============================================================
# Visualize data in 2D using first two principal components
# ============================================================
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled_all)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], 
                       c=df_featured['SalePrice'], 
                       cmap='RdYlGn', alpha=0.6, s=20)
plt.colorbar(scatter, label='Sale Price ($)')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Ames Housing Data — First 2 Principal Components', fontweight='bold')
plt.show()

print("Notice how price roughly increases along PC1 — this principal component")
print("has captured the primary direction that relates to house value.")

In [ ]:
# ============================================================
# Does PCA help or hurt model performance?
# ============================================================
y = df_featured['SalePrice']

results = {}

scores_full = cross_val_score(LinearRegression(), X_scaled_all, y, cv=5, scoring='r2')
results[f'All {X_scaled_all.shape[1]} features'] = scores_full.mean()

for n_comp in [5, 10, n_90, n_95]:
    if n_comp >= X_scaled_all.shape[1]:
        continue
    pca = PCA(n_components=n_comp)
    X_pca = pca.fit_transform(X_scaled_all)
    scores = cross_val_score(LinearRegression(), X_pca, y, cv=5, scoring='r2')
    var_explained = pca.explained_variance_ratio_.sum()
    results[f'PCA ({n_comp} comp, {var_explained:.0%} var)'] = scores.mean()

print("Linear Regression R² with different feature sets:")
print("=" * 55)
for name, score in results.items():
    print(f"  {name:<42s} {score:.4f}")

### PCA Takeaway

PCA is not always a win for prediction — you might lose important information. It's most useful when:
- You have many correlated features
- You need to reduce computation time
- You want to visualize high-dimensional data
- You're fighting overfitting with too many features relative to samples

For pyMAISE applications, PCA can be valuable when dealing with large numbers of spatial or spectral measurements that are highly correlated.

---

## Part 7: Feature Selection

The opposite of creating features — **choosing which features to keep**.

### Why Select?
- Reduce overfitting
- Improve interpretability
- Speed up training
- Remove noisy or irrelevant features

### Three Approaches:

1. **Filter methods:** Statistical tests to score features independently (fast, model-agnostic)
2. **Wrapper methods:** Use model performance to evaluate feature subsets (slow, thorough)
3. **Embedded methods:** Feature importance from the model itself (e.g., Random Forest importances)

In [ ]:
# ============================================================
# Method 1: Correlation Analysis (Filter Method)
# ============================================================
all_numeric_with_target = df_featured.select_dtypes(include=[np.number]).drop(
    columns=['Id'], errors='ignore'
)
correlations = all_numeric_with_target.corr()['SalePrice'].drop('SalePrice').sort_values()

n_show = 20
top_corr = pd.concat([correlations.head(n_show // 2), correlations.tail(n_show // 2)])

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in top_corr.values]
top_corr.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Correlation with SalePrice')
ax.set_title('Top Feature Correlations with Sale Price', fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("Top 20 most correlated features (by |r|):")
for feat, corr in correlations.abs().sort_values(ascending=False).head(20).items():
    sign = '+' if correlations[feat] > 0 else '-'
    print(f"  {feat:<25s} r = {sign}{corr:.3f}")

In [ ]:
# ============================================================
# Check for multicollinearity between features
# ============================================================
feature_corr = all_numeric_with_target.drop(columns=['SalePrice']).corr()

high_corr_pairs = []
for i in range(len(feature_corr.columns)):
    for j in range(i+1, len(feature_corr.columns)):
        if abs(feature_corr.iloc[i, j]) > 0.8:
            high_corr_pairs.append((
                feature_corr.columns[i], 
                feature_corr.columns[j], 
                feature_corr.iloc[i, j]
            ))

print(f"Highly correlated feature pairs (|r| > 0.8): {len(high_corr_pairs)} found")
print("=" * 65)
for f1, f2, corr in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True)[:15]:
    print(f"  {f1:<25s} ↔ {f2:<25s}  r = {corr:+.3f}")

print("\n These pairs contain redundant information.")
print("   Consider keeping only one from each highly correlated pair.")
print("   (e.g., GarageCars and GarageArea tell you roughly the same thing)")

In [ ]:
# ============================================================
# Method 2: SelectKBest (Filter Method with statistical test)
# ============================================================
X_all = all_numeric_with_target.drop(columns=['SalePrice'])
y = df_featured['SalePrice']

selector = SelectKBest(score_func=f_regression, k=15)
X_selected = selector.fit_transform(X_all, y)

scores_df = pd.DataFrame({
    'Feature': X_all.columns,
    'F-Score': selector.scores_,
    'p-value': selector.pvalues_,
    'Selected': selector.get_support()
}).sort_values('F-Score', ascending=False)

print("SelectKBest Results (top 20):")
print("=" * 65)
for _, row in scores_df.head(20).iterrows():
    marker = "✅" if row['Selected'] else "  "
    print(f"  {marker} {row['Feature']:<25s} F={row['F-Score']:>10.1f}  p={row['p-value']:.2e}")

In [ ]:
# ============================================================
# Method 3: Random Forest Feature Importance (Embedded Method)
# ============================================================
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_all, y)

importances = pd.DataFrame({
    'Feature': X_all.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
importances.tail(20).plot(kind='barh', x='Feature', y='Importance', 
                           ax=ax, color='steelblue', legend=False)
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)')
ax.set_title('Random Forest Feature Importances (Top 20)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Compare: All features vs. Selected features
# ============================================================
top_10_features = importances.tail(10)['Feature'].tolist()
print(f"Top 10 features selected by Random Forest:")
for i, feat in enumerate(reversed(top_10_features), 1):
    imp = importances.set_index('Feature').loc[feat, 'Importance']
    print(f"  {i:>2d}. {feat:<25s} importance = {imp:.4f}")

X_all_scaled = StandardScaler().fit_transform(X_all)
X_top10_scaled = StandardScaler().fit_transform(X_all[top_10_features])

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

print(f"\nModel Performance: All {X_all.shape[1]} Features vs. Top 10")
print("=" * 65)

for name, model in models.items():
    scores_all = cross_val_score(model, X_all_scaled, y, cv=5, scoring='r2')
    scores_top = cross_val_score(model, X_top10_scaled, y, cv=5, scoring='r2')
    
    print(f"\n{name}:")
    print(f"  All {X_all.shape[1]} features: R² = {scores_all.mean():.4f} ± {scores_all.std():.4f}")
    print(f"  Top 10 features:  R² = {scores_top.mean():.4f} ± {scores_top.std():.4f}")

---

## Part 8: Putting It All Together — sklearn Pipelines

In practice, you don't do these steps manually one at a time. Scikit-learn's `Pipeline` lets you chain preprocessing steps and a model into a single object that:

1. **Prevents data leakage** — each step fits only on training data
2. **Simplifies code** — one `.fit()` and `.predict()` call
3. **Works with cross-validation** — the entire pipeline is cross-validated correctly

This is how professionals structure their ML code.

In [ ]:
# ============================================================
# Build a complete pipeline starting from the cleaned data
# ============================================================

# Start from df_clean and add our best engineered features
df_pipe = df_clean.copy()
df_pipe['HouseAge'] = df_pipe['YrSold'] - df_pipe['YearBuilt']
df_pipe['TotalSF'] = df_pipe['GrLivArea'] + df_pipe['TotalBsmtSF']
df_pipe['TotalBath'] = (df_pipe['FullBath'] + 0.5 * df_pipe['HalfBath'] + 
                         df_pipe['BsmtFullBath'] + 0.5 * df_pipe['BsmtHalfBath'])
df_pipe['QualityArea'] = df_pipe['OverallQual'] * df_pipe['GrLivArea']

# Define feature groups
numeric_features = ['LotArea', 'GrLivArea', 'OverallQual', 'OverallCond',
                     'TotalBsmtSF', 'BedroomAbvGr', 'GarageCars', 'GarageArea',
                     'HouseAge', 'TotalSF', 'TotalBath', 'QualityArea']

categorical_features = ['Neighborhood', 'HouseStyle', 'SaleCondition', 
                         'MSZoning', 'BldgType']

# Verify all columns exist
for col in numeric_features + categorical_features:
    assert col in df_pipe.columns, f"Column '{col}' not found!"
print("All columns verified ")

# Build the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(drop='first', sparse_output=False, 
                                      handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

# Complete pipelines
pipe_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

pipe_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

print("\nPipeline structure:")
print(pipe_lr)

In [ ]:
# ============================================================
# Train and evaluate — the CORRECT way
# ============================================================
X = df_pipe[numeric_features + categorical_features]
y = df_pipe['SalePrice']

# Split FIRST, then let the pipeline handle everything
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print()

for name, pipe in [('Linear Regression', pipe_lr), ('Random Forest', pipe_rf)]:
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2')
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    test_r2 = r2_score(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"{name}:")
    print(f"  CV R² (train):  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  Test R²:        {test_r2:.4f}")
    print(f"  Test RMSE:      ${test_rmse:,.0f}")
    print()

In [ ]:
# ============================================================
# Visualize predictions vs actual
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (name, pipe) in zip(axes, [('Linear Regression', pipe_lr), ('Random Forest', pipe_rf)]):
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    ax.scatter(y_test, y_pred, alpha=0.5, s=20, color='steelblue')
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
            'r--', linewidth=2, label='Perfect Prediction')
    ax.set_xlabel('Actual Sale Price ($)', fontsize=11)
    ax.set_ylabel('Predicted Sale Price ($)', fontsize=11)
    ax.set_title(f'{name}\nR² = {r2_score(y_test, y_pred):.4f}', 
                 fontsize=13, fontweight='bold')
    ax.legend()

plt.suptitle('Predicted vs Actual Sale Price (Test Set)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Summary & Cheat Sheet

| Step | What | Key Tools | When |
|---|---|---|---|
| **Missing Data** | Handle NaN values | `SimpleImputer`, domain logic | Always first |
| **Encoding** | Convert categories to numbers | `OneHotEncoder`, `OrdinalEncoder` | When you have text columns |
| **Scaling** | Normalize feature ranges | `StandardScaler`, `MinMaxScaler` | Before linear models, SVMs, NNs, PCA |
| **Feature Creation** | Build new features | Domain knowledge, `PolynomialFeatures` | When you understand the problem |
| **Dimensionality Reduction** | Compress feature space | `PCA` | Many correlated features |
| **Feature Selection** | Choose best features | `SelectKBest`, RF importances, correlation | Reduce overfitting |
| **Pipeline** | Chain it all together | `Pipeline`, `ColumnTransformer` | Always in production |

### Rules of Thumb

1. **Always explore your data first** — `df.describe()`, `df.isnull().sum()`, correlation plots
2. **Think about WHY data is missing** before deciding how to fill it
3. **Scale before algorithms that use distance or gradients** (linear models, SVMs, NNs, KNN)
4. **Don't scale for tree-based models** (they don't need it)
5. **Use Pipelines** to prevent data leakage and keep code clean
6. **Domain knowledge > automated methods** for feature creation
7. **More features ≠ better model** — sometimes less is more